# NewsBot Intelligence System 2.0 — 04 Language Models and Retrieval

## Goal
Demonstrate summary fallbacks, quality diagnostics, retrieval, related content, and grounded enhancement.

This notebook imports reusable project modules rather than duplicating implementation logic.

## Summarization paths
The default uses a tested extractive path. Set NEWSBOT_ENABLE_TRANSFORMERS=1 to opt into the lazy pretrained model.

In [1]:
from src.language_models import IntelligentSummarizer, SemanticSearchEngine
from src.data_processing.data_validator import load_news_dataset
summarizer = IntelligentSummarizer()
text = 'The city announced new public health measures after a heat wave. Officials said cooling centers would open and residents should check on vulnerable neighbors. The measures start this weekend.'
summary = summarizer.summarize_article(text)
{'summary': summary, 'status': summarizer.backend_status(), 'quality': summarizer.assess_summary_quality(text, summary)}

{'summary': 'The city announced new public health measures after a heat wave. Officials said cooling centers would open and residents should check on vulnerable neighbors. The measures start this weekend.',
 'status': {'backend': 'extractive',
  'model': None,
  'transformer_requested': False,
  'warning': None},
 'quality': {'compression_ratio': 1.0,
  'summary_words': 29,
  'length_within_target': True,
  'flesch_reading_ease': 48.24,
  'entity_preservation_proxy': 1.0,
  'backend': 'extractive'}}

## Multi-document summary
Short corpus records are combined for a more meaningful demonstration.

In [2]:
df = load_news_dataset()
records = df[df.category == 'TECH'].head(4).to_dict('records')
summarizer.summarize_multiple_articles(records, focus_topic='technology')

'Tweets from the community are helpful to get a distributed feel of the situation but they have to be taken with a grain (a byte?) of salt. The Battle for the Future of the Internet? Singing from the same songbook as Web juggernaut Google, the U.S.'

## Semantic retrieval
The active backend is disclosed; relatedness is not factual agreement.

In [3]:
search = SemanticSearchEngine().build_index(df.full_text, df[['article_id','title','category','date']].to_dict('records'))
results = search.semantic_search('artificial intelligence software businesses', 5)
{'backend': search.backend_status(), 'results': [{k:v for k,v in row.items() if k in ['article_id','title','category','score','retrieval_backend']} for row in results], 'expanded_query': search.expand_query('artificial intelligence software businesses', results)}

{'backend': {'backend': 'tfidf',
  'model': None,
  'transformer_requested': False,
  'warning': None},
 'results': [{'article_id': 619,
   'title': 'How To Stop Worrying And Love Artificial Intelligence',
   'category': 'TECH',
   'score': 0.2861093740780504,
   'retrieval_backend': 'tfidf'},
  {'article_id': 1354,
   'title': 'In a Huge Breakthrough, Google’s AI Beats a Top Player at the Game of Go',
   'category': 'TECH',
   'score': 0.22030456486347547,
   'retrieval_backend': 'tfidf'},
  {'article_id': 1166,
   'title': "Autism Without Fear: Is Corporate Use of 'Emotional Intelligence' Grounds for Discrimination Under the ADA?",
   'category': 'BUSINESS',
   'score': 0.10403869759349639,
   'retrieval_backend': 'tfidf'},
  {'article_id': 1040,
   'title': 'Trump Distances Himself From His Own Remarks On Russian Election Meddling',
   'category': 'POLITICS',
   'score': 0.07451770944898113,
   'retrieval_backend': 'tfidf'},
  {'article_id': 582,
   'title': "Mike Lynch, Autonomy Fo

## Evaluation evidence
Summary and retrieval metrics are small authored tests, not a production benchmark.

In [4]:
import json
from pathlib import Path
metrics = Path('data/results/metrics')
{file.name: json.loads(file.read_text()) for file in [metrics/'summarization_evaluation.json', metrics/'semantic_search_evaluation.json']}

{'summarization_evaluation.json': {'examples': 1,
  'backend': {'backend': 'extractive',
   'model': None,
   'transformer_requested': False,
   'warning': None},
  'mean': {'compression_ratio': 0.4936708860759494,
   'flesch_reading_ease': 35.29,
   'entity_preservation_proxy': 0.6666666666666666,
   'rougeL': 0.2033898305084746}},
 'semantic_search_evaluation.json': {'precision_at_1': 0.6666666666666666,
  'precision_at_5': 0.7333333333333334,
  'hit_rate_at_5': 1.0,
  'queries': 6,
  'note': 'Category-based authored relevance evaluation; it measures topical retrieval, not factual agreement.'}}

## Limitations and ethics
The source corpus is historical and predominantly English. Confidence is not truth; sentiment, topics, named entities, translation, summaries, and semantic similarity can be wrong. Entity co-occurrence is not a proven real-world relationship. Internal corpus corroboration is not independent fact-checking.